# 作业 02：把多个输入接入上周的训练模板

作业 01 比较了同一个样本的两种表示：两个标量输入张量，或一个包含两个数的输入张量。本题把多个样本分别放入 x1、x2，练习补全训练代码。

请参考上周模板，让同一个预测函数同时使用 x1 和 x2。所有样本共享 w1、w2、b；每次预测都需要两个输入。下面提供函数和数据，参数创建、损失、梯度与更新由你完成。

本文件可以独立运行，不依赖作业 01 的内核变量。


In [1]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b


In [4]:
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

learning_rate = 0.01
epochs = 1000

for epoch in range(epochs):
    y_pred = predict(x1, x2, w1, w2, b)
    loss = ((y_pred - y) ** 2).mean()
    
    if w1.grad is not None:
        w1.grad.zero_()
        w2.grad.zero_()
        b.grad.zero_()
        
    loss.backward()
    
    with torch.no_grad():
        w1 -= learning_rate * w1.grad
        w2 -= learning_rate * w2.grad
        b -= learning_rate * b.grad
        
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 100, Loss: 0.2981
Epoch 200, Loss: 0.0346
Epoch 300, Loss: 0.0041
Epoch 400, Loss: 0.0005
Epoch 500, Loss: 0.0001
Epoch 600, Loss: 0.0000
Epoch 700, Loss: 0.0000
Epoch 800, Loss: 0.0000
Epoch 900, Loss: 0.0000
Epoch 1000, Loss: 0.0000


## 训练数据：每一行是同一个样本的两个输入和一个目标

下面 x1、x2、y 的 shape 都为 `(6,)`，表示 6 个样本。`predict(x1, x2, ...)` 同时使用两个输入张量，返回 6 个预测值。

| 样本 i | x1[i] | x2[i] | y[i] |
|---|---|---|---|
| 0 | -2 | 1 | -6 |
| 1 | -1 | 0 | -1 |
| 2 | 0 | 2 | -5 |
| 3 | 1 | -1 | 6 |
| 4 | 2 | 3 | -4 |
| 5 | 3 | 1 | 4 |

训练时不要修改输入和目标，也不要直接填写猜测的参数答案。


In [3]:
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y = torch.tensor([-6., -1., -5., 6., -4., 4.])
print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y, 'shape:', tuple(y.shape))


x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.]) shape: (6,)


## 你的训练区：修改上周模板

创建三个浮点标量参数 w1、w2、b，都启用梯度，并一起交给优化器。每轮调用给定的 predict 函数，同时传入 x1、x2，得到 shape 为 `(6,)` 的 y_pred。

使用绝对值损失：先得到 `(6,)` 的逐样本误差，再用 `.mean()` 得到 shape 为 `()` 的平均损失。只对这个平均损失 backward 一次，然后更新全部参数。

可以从三个参数都为 0、2000 轮、学习率 `0.1 / (epoch ** 0.5)` 开始。这里预测公式是线性的，零初始化与上周的平方参数例子不同。

补完后删除 NotImplementedError。


In [6]:
import torch.optim as optim

w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

optimizer = optim.SGD([w1, w2, b], lr=0.1)

epochs = 2000

for epoch in range(epochs):
    current_lr = 0.1 / (epoch ** 0.5) if epoch > 0 else 0.1
    for param_group in optimizer.param_groups:
        param_group['lr'] = current_lr

    y_pred = predict(x1, x2, w1, w2, b)
    
    loss = torch.abs(y - y_pred).mean()
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.6f}")

print(f"\n最终参数：w1={w1.item():.4f}, w2={w2.item():.4f}, b={b.item():.4f}")

Epoch 200, Loss: 0.707411
Epoch 400, Loss: 0.357061
Epoch 600, Loss: 0.089920
Epoch 800, Loss: 0.003316
Epoch 1000, Loss: 0.002791
Epoch 1200, Loss: 0.002406
Epoch 1400, Loss: 0.002104
Epoch 1600, Loss: 0.001862
Epoch 1800, Loss: 0.001658
Epoch 2000, Loss: 0.001492

最终参数：w1=2.0026, w2=-2.9990, b=1.0005


## 我的观察与实验

1. 现在一次预测需要几个输入张量？x1 和 x2 与样本数 6 分别是什么关系？（10分）
2. 对照上周：预测公式、参数列表和损失计算分别改了哪里？优化器的基本步骤变了吗？（10分）
3. 记录最终参数、预测和平均损失，说明如何判断训练有效。（20分）
4. 为什么这次 w1=w2=b=0 仍然可以开始学习？（10分）

待填写。


In [ ]:
1.两个；x1和x2都包含着6个样本的特征值
2.w2 * (x**2)变成了w2 * x2,x3变成b了；参数列表多加了一个b；损失计算不变；优化器的基本步骤不变
3.w1 = 2.0026, w2 = 2.9990, b = 1.005;平均损失为0.001492；从最初的0.707降到0.001492，非常接近0，说明模型预测越来越准；
4.pytorch能根据误差，计算参数。